In [1]:
import sys
import cv2
import numpy as np
from tensorflow.keras.models import load_model

from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QPushButton, QVBoxLayout, QFileDialog, QMessageBox
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import QTimer


class FireClassifier:
    def __init__(self, model_path):
        self.model = load_model(model_path)

        self.img_size = 128

        self.classes = ["Fire", "No Fire"]

    def preprocess(self, img):
        img_resized = cv2.resize(img, (self.img_size, self.img_size))
        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

        img_norm = img_rgb / 255.0

        img_input = np.expand_dims(img_norm, axis=0)

        return img_input

    def predict(self, img):

        input_img = self.preprocess(img)

        prediction = self.model.predict(input_img, verbose=0)

        confidence = float(prediction[0][0])

        if confidence > 0.5:
            label = "No Fire"
            color = (0, 0, 255)
            score = confidence
        else:
            label = "Fire"
            color = (0, 255, 0)
            score = 1 - confidence

        text = f"{label} : {score:.2f}"

        cv2.putText(
            img,
            text,
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            color,
            2
        )

        return img


class FireApp(QWidget):

    def __init__(self, classifier):
        super().__init__()

        self.classifier = classifier

        self.setWindowTitle("Détection Feu / Pas Feu")
        self.setGeometry(100, 100, 800, 600)

        self.cap = None

        self.timer = QTimer()
        self.timer.timeout.connect(self.update_frame)

        self.setup_ui()

    def setup_ui(self):

        layout = QVBoxLayout()

        self.image_label = QLabel()
        layout.addWidget(self.image_label)

        self.btn_image = QPushButton("Charger Image")
        self.btn_image.clicked.connect(self.load_image)
        layout.addWidget(self.btn_image)

        self.btn_webcam = QPushButton("Démarrer Webcam")
        self.btn_webcam.clicked.connect(self.start_webcam)
        layout.addWidget(self.btn_webcam)

        self.setLayout(layout)

    def load_image(self):

        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Choisir une image",
            "",
            "Images (*.png *.jpg *.jpeg)"
        )

        if not file_path:
            return

        img = cv2.imread(file_path)

        if img is None:
            QMessageBox.warning(self, "Erreur", "Image invalide")
            return

        result = self.classifier.predict(img)

        self.display_image(result)

    def start_webcam(self):

        if self.cap is None:

            self.cap = cv2.VideoCapture(0)

            if not self.cap.isOpened():
                QMessageBox.critical(
                    self,
                    "Erreur",
                    "Impossible d'ouvrir la webcam"
                )
                return

            self.timer.start(30)

    def update_frame(self):

        ret, frame = self.cap.read()

        if ret:
            result = self.classifier.predict(frame)
            self.display_image(result)

    def display_image(self, img):

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, ch = img_rgb.shape

        bytes_per_line = ch * w

        qimg = QImage(
            img_rgb.data,
            w,
            h,
            bytes_per_line,
            QImage.Format.Format_RGB888
        )

        pixmap = QPixmap.fromImage(qimg)

        self.image_label.setPixmap(
            pixmap.scaled(
                self.image_label.width(),
                self.image_label.height()
            )
        )

    def closeEvent(self, event):

        if self.cap:
            self.cap.release()

        event.accept()

if __name__ == "__main__":

    MODEL_PATH = r"C:\Users\Ibrahim Manar\Desktop\DL Project V2\best_cnn_model.h5"

    classifier = FireClassifier(MODEL_PATH)

    app = QApplication(sys.argv)

    window = FireApp(classifier)

    window.show()

    sys.exit(app.exec())

SystemExit: 0

C:\Users\Ibrahim Manar\anaconda_3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
pyinstaller --onefile --windowed --add-data "model.h5;." app.py

In [ ]:
C:\Users\Ibrahim Manar\Desktop\DL Project V2\best_cnn_model.h5